In [28]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import root_mean_squared_error

In [2]:
df=pd.DataFrame(pd.read_csv("customer_churn.csv"))

In [3]:
df

,Age,Tenure,MonthlyCharges,Contract,InternetService,PaymentMethod,SupportCalls,Churn
0,23,53,142.05,Month-to-month,Fiber optic,Mailed check,9,No
1,65,64,143.19,Month-to-month,Fiber optic,Credit card,3,No
2,58,41,89.68,Two year,Fiber optic,Mailed check,8,No
3,45,30,52.61,One year,Fiber optic,Bank transfer,0,No
4,44,0,116.11,One year,Fiber optic,Mailed check,7,Yes
...,...,...,...,...,...,...,...,...
99995,79,40,43.50,Month-to-month,Fiber optic,Bank transfer,6,Yes
99996,19,25,100.76,Month-to-month,DSL,Electronic check,2,No
99997,53,8,45.82,Month-to-month,DSL,Bank transfer,6,No
99998,18,65,34.24,Month-to-month,DSL,Credit card,0,No


In [6]:
X=df.drop("Churn",axis=1).copy()
y=df["Churn"]

In [7]:
X

,Age,Tenure,MonthlyCharges,Contract,InternetService,PaymentMethod,SupportCalls
0,23,53,142.05,Month-to-month,Fiber optic,Mailed check,9
1,65,64,143.19,Month-to-month,Fiber optic,Credit card,3
2,58,41,89.68,Two year,Fiber optic,Mailed check,8
3,45,30,52.61,One year,Fiber optic,Bank transfer,0
4,44,0,116.11,One year,Fiber optic,Mailed check,7
...,...,...,...,...,...,...,...
99995,79,40,43.50,Month-to-month,Fiber optic,Bank transfer,6
99996,19,25,100.76,Month-to-month,DSL,Electronic check,2
99997,53,8,45.82,Month-to-month,DSL,Bank transfer,6
99998,18,65,34.24,Month-to-month,DSL,Credit card,0


In [8]:
y

0         No
1         No
2         No
3         No
4        Yes
        ... 
99995    Yes
99996     No
99997     No
99998     No
99999     No
Name: Churn, Length: 100000, dtype: object

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20,random_state=42,stratify=y)

In [12]:
numeric_cols = X_train.select_dtypes(
    include=np.number
).columns.tolist()

categorical_cols = X_train.select_dtypes(
    exclude=np.number
).columns.tolist()

print("Numerical:", numeric_cols)
print("Categorical:", categorical_cols)

Numerical: ['Age', 'Tenure', 'MonthlyCharges', 'SupportCalls']
Categorical: ['Contract', 'InternetService', 'PaymentMethod']


In [13]:
numeric_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median")),("scaler", StandardScaler())])

In [16]:
numeric_pipeline

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())])

In [14]:
categorical_pipeline = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),("Encoder", OneHotEncoder(sparse_output=False))])

In [17]:
categorical_pipeline

Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')),
                ('Encoder', OneHotEncoder(sparse_output=False))])

In [25]:
Complete_Pipeline = ColumnTransformer([("number",numeric_pipeline,X_train.select_dtypes(include=np.number).columns.tolist()),("category",categorical_pipeline,
                    X_train.select_dtypes(exclude=np.number).columns.tolist())],verbose_feature_names_out=False,verbose=True).set_output(transform="pandas")

In [26]:
Complete_Pipeline

ColumnTransformer(transformers=[('number',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['Age', 'Tenure', 'MonthlyCharges',
                                  'SupportCalls']),
                                ('category',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('Encoder',
                                                  OneHotEncoder(sparse_output=False))]),
                                 ['Contract', 'InternetService',
                                  'PaymentMethod'])],
                  verbose=True, verbose_feature_names_out=False)

In [29]:
model = LogisticRegression(max_iter=1000,random_state=42)

In [30]:
model

LogisticRegression(max_iter=1000, random_state=42)

In [32]:
model_pipeline = Pipeline([("preprocessor",Complete_Pipeline),("model", model)])

In [33]:
model_pipeline

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('number',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age', 'Tenure',
                                                   'MonthlyCharges',
                                                   'SupportCalls']),
                                                 ('category',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('Encoder',
                                                                   OneHotEncoder(sparse_output=False))]),
                                                  ['Contract',
                                                   'InternetService',
                                                   'PaymentMethod'])],
                                   verbose=True,
                                   verbose_feature_names_out=False)),
                ('model', LogisticRegression(max_iter=1000, random_state=42))])

In [39]:
model_pipeline.fit(X_train, y_train)

[ColumnTransformer] ........ (1 of 2) Processing number, total=   0.1s
[ColumnTransformer] ...... (2 of 2) Processing category, total=   0.2s


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('number',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age', 'Tenure',
                                                   'MonthlyCharges',
                                                   'SupportCalls']),
                                                 ('category',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('Encoder',
                                                                   OneHotEncoder(sparse_output=False))]),
                                                  ['Contract',
                                                   'InternetService',
                                                   'PaymentMethod'])],
                                   verbose=True,
                                   verbose_feature_names_out=False)),
                ('model', LogisticRegression(max_iter=1000, random_state=42))])

In [34]:
cv = StratifiedKFold(n_splits=5,shuffle=True,random_state=42)

In [36]:
cv_accuracy = cross_val_score(model_pipeline,X_train,y_train,cv=cv,scoring="accuracy",n_jobs=-1)

In [37]:
cv_accuracy

array([0.733125, 0.733125, 0.733125, 0.733125, 0.733125])

In [38]:
cv_accuracy.mean()

0.733125

In [40]:
y_pred = model_pipeline.predict(X_test)

In [41]:
y_pred

array(['No', 'No', 'No', ..., 'No', 'No', 'No'], dtype=object)